In [1]:
import os
from ultralytics import YOLO
from tqdm import tqdm

In [2]:
MODEL_PATH = 'runs/detect/KITTI_Object_Detection/yolo26_kitti_experiment/weights/best.pt'
VAL_IMAGES_DIR = 'kitti_yolo/images/val'
RESULTS_OUTPUT_DIR = 'kitti_results/data'
INV_CLASS_MAP = {0: 'Car', 1: 'Pedestrian', 2: 'Cyclist'}

In [3]:
model = YOLO(MODEL_PATH)
os.makedirs(RESULTS_OUTPUT_DIR, exist_ok=True)

image_files = [f for f in os.listdir(VAL_IMAGES_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

for img_name in tqdm(image_files):
    img_path = os.path.join(VAL_IMAGES_DIR, img_name)

    results = model.predict(source=img_path, conf=0.25, imgsz=640, verbose=False)[0]

    txt_name = os.path.splitext(img_name)[0] + ".txt"
    result_path = os.path.join(RESULTS_OUTPUT_DIR, txt_name)

    with open(result_path, 'w') as f:
        for box in results.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            xyxy = box.xyxy[0].tolist()
            # Map ID back to String
            label = INV_CLASS_MAP.get(cls_id, 'Misc')

            # KITTI 16-COLUMN FORMAT:
            # 1: Type (e.g., 'Car')
            # 2: Truncated (0.0)
            # 3: Occluded (0)
            # 4: Alpha (0.0)
            # 5-8: 2D Bbox (left, top, right, bottom)
            # 9-11: 3D Dimensions (0,0,0)
            # 12-14: 3D Location (0,0,0)
            # 15: Rotation_y (0.0)
            # 16: Score (Confidence)

            kitti_row = (
                f"{label} 0.00 0 0.00 "
                f"{xyxy[0]:.2f} {xyxy[1]:.2f} {xyxy[2]:.2f} {xyxy[3]:.2f} "
                f"0.00 0.00 0.00 0.00 0.00 0.00 0.00 {conf:.4f}\n"
            )
            f.write(kitti_row)

🚀 Starting inference on images in: kitti_yolo/images/val


100%|██████████| 748/748 [00:13<00:00, 57.45it/s]


✅ Done! 748 files processed.
📍 Results saved to: /home/pouya/OneDrive/Visual Perception/Project/kitti_results/data
